# 특이값 분해 (SVD) - SVD의 원리와 응용

- Tutorial ID: `math-4-1`
- Tutorial: 특이값 분해 (SVD)
- Section ID: `math-4-1-1`
- Section: SVD의 원리와 응용


# 058 | 특이값 분해 (SVD, Singular Value Decomposition)

> **대상**: 선형대수 기초를 배운 후, 딥러닝/트랜스포머에 어떻게 쓰이는지 궁금한 분  
> **목표**: SVD의 수학적 원리를 이해하고, NumPy로 직접 계산하며, 트랜스포머(LoRA, Attention)에서의 응용을 실습한다.

---

## 📌 목차
1. SVD란 무엇인가? (직관적 이해)
2. SVD 수학 공식: A = U Σ Vᵀ
3. NumPy로 SVD 직접 계산해보기
4. 저랭크 근사 (Low-Rank Approximation) — 이미지 압축 실습
5. 트랜스포머에서 SVD: LoRA (Low-Rank Adaptation) 원리
6. Attention 가중치 행렬의 SVD 분석
7. 연습 문제

---
## 1. SVD란 무엇인가? — 직관적 이해

### 행렬을 '분해'한다는 것은?

어떤 행렬 **A**가 있을 때, 우리는 이 행렬을 세 가지 단순한 행렬의 곱으로 나눌 수 있습니다:

$$A = U \Sigma V^T$$

| 행렬 | 의미 | 특성 |
|------|------|------|
| **U** | 입력 공간의 방향 (왼쪽 특이벡터) | 열이 서로 직교 (orthogonal) |
| **Σ** | 각 방향의 '중요도' (특이값) | 대각행렬, 값이 크면 더 중요 |
| **Vᵀ** | 출력 공간의 방향 (오른쪽 특이벡터) | 행이 서로 직교 |

### 비유: 사진을 압축하는 것처럼

사진 데이터(픽셀 행렬)를 SVD로 분해하면, **중요한 정보(큰 특이값)**만 남기고  
**덜 중요한 정보(작은 특이값)**는 버려서 파일 크기를 줄일 수 있습니다.  
이것이 바로 **저랭크 근사(Low-Rank Approximation)**입니다!

In [ ]:
# ============================================================
# 필요한 라이브러리 임포트

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# 한글 폰트 설정 (Mac: 'AppleGothic', Linux: 'NanumGothic', Windows: 'Malgun Gothic')
# 한글이 깨지면 아래 폰트를 시스템에 맞게 변경하세요
# Colab/로컬/브라우저 환경에서 한글 폰트가 없을 때 자동으로 NanumGothic을 등록합니다.
# (Colab 기본 런타임에는 NanumGothic이 없어서 matplotlib 한글 glyph 경고가 발생할 수 있습니다.)
import os
import urllib.request
import matplotlib.font_manager as fm

font_path = '/tmp/NanumGothic-Regular.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve(
            'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf',
            font_path,
        )
    fm.fontManager.addfont(font_path)
    matplotlib.rcParams['font.family'] = ['NanumGothic', 'DejaVu Sans']
except Exception as e:
    print(f'한글 폰트 설정 경고: {e}')

plt.rcParams['axes.unicode_minus'] = False

np.set_printoptions(precision=3, suppress=True)  # 소수점 3자리, 과학적 표기 제거
print('라이브러리 로드 완료!')

---
## 2. SVD 수학 공식: A = U Σ Vᵀ

### 간단한 예제로 손으로 따라가기

아래 행렬을 생각해봅시다:

$$A = \begin{bmatrix} 3 & 1 \\ 1 & 3 \\ 1 & 1 \end{bmatrix}$$

이 행렬은 3행 2열짜리(3×2)입니다.  
SVD로 분해하면:
- U: 3×3 직교행렬  
- Σ: 3×2 대각행렬  
- Vᵀ: 2×2 직교행렬

In [ ]:
# ============================================================
# SVD 기본 계산 예제

In [ ]:
# 간단한 3x2 행렬 정의
A = np.array([
    [3, 1],
    [1, 3],
    [1, 1]
], dtype=float)

print('원본 행렬 A:')
print(A)
print(f'행렬 크기: {A.shape}  (행={A.shape[0]}, 열={A.shape[1]})')

# np.linalg.svd()로 SVD 수행
# full_matrices=True → U는 3x3, V는 2x2 (full SVD)
# full_matrices=False → U는 3x2, V는 2x2 (thin/compact SVD, 더 실용적)
U, S, Vt = np.linalg.svd(A, full_matrices=False)

print('\n--- SVD 결과 ---')
print(f'U (왼쪽 특이벡터): shape={U.shape}')
print(U)

print(f'\nS (특이값 벡터): shape={S.shape}')
print(S)
# 주의: numpy는 S를 대각행렬이 아닌 1D 벡터로 반환합니다
# 대각행렬로 보려면 np.diag(S) 사용

print(f'\nVt (오른쪽 특이벡터, V의 전치행렬): shape={Vt.shape}')
print(Vt)

# 검증: U @ diag(S) @ Vt ≈ 원본 A?
A_reconstructed = U @ np.diag(S) @ Vt
print('\n--- 검증: U @ Σ @ Vᵀ로 복원한 행렬 ---')
print(A_reconstructed)
print(f'원본 A와의 최대 오차: {np.max(np.abs(A - A_reconstructed)):.2e}')
print('→ 거의 0에 가까우면 SVD가 제대로 된 것!')

In [ ]:
# ============================================================
# 특이값의 의미: 각 성분이 얼마나 '중요한가'?

In [ ]:
print('특이값(Singular Values):', S)
print('큰 특이값 = 더 중요한 정보를 담은 방향')
print()

# 특이값의 제곱합 대비 각 성분의 기여도
total_variance = np.sum(S**2)
for i, s in enumerate(S):
    contribution = (s**2 / total_variance) * 100
    print(f'  σ_{i+1} = {s:.3f}  →  전체 정보의 {contribution:.1f}% 설명')

# 시각화: 특이값의 크기
plt.figure(figsize=(6, 4))
plt.bar(range(1, len(S)+1), S, color=['#4CAF50', '#2196F3'])
plt.xlabel('특이값 번호')
plt.ylabel('특이값 크기')
plt.title('특이값(σ) 크기 비교')
plt.xticks(range(1, len(S)+1), [f'σ_{i}' for i in range(1, len(S)+1)])
plt.tight_layout()
plt.show()

---
## 3. 직교성(Orthogonality) 확인

SVD에서 U와 V는 **직교행렬(orthogonal matrix)**입니다.  
직교행렬의 핵심 성질: **Uᵀ × U = I** (단위행렬)

이는 각 열벡터들이 서로 수직(내적=0)이고, 크기가 1임을 의미합니다.

In [ ]:
# ============================================================
# 직교성 검증

In [ ]:
print('=== U의 직교성 확인 ===')  
UtU = U.T @ U
print('Uᵀ @ U (단위행렬에 가까워야 함):')
print(UtU)
print(f'단위행렬과의 최대 차이: {np.max(np.abs(UtU - np.eye(UtU.shape[0]))):.2e}')

print()
print('=== V의 직교성 확인 ===')
VVt = Vt.T @ Vt  # V = Vt.T 이므로 V.T @ V = Vt @ Vt.T
print('Vᵀ @ V (단위행렬에 가까워야 함):')
print(VVt)
print(f'단위행렬과의 최대 차이: {np.max(np.abs(VVt - np.eye(VVt.shape[0]))):.2e}')

print()
print('▶ 두 값 모두 ~0에 가까우면 SVD의 U, V가 올바른 직교행렬임을 의미합니다!')

---
## 4. 저랭크 근사 (Low-Rank Approximation) — 이미지 압축 실습

SVD의 핵심 응용 중 하나는 **저랭크 근사**입니다.  
큰 특이값 k개만 사용하면 원본 행렬을 '충분히 잘' 표현할 수 있습니다:

$$A_k = \sum_{i=1}^{k} \sigma_i \cdot u_i \cdot v_i^T$$

여기서 각 `σᵢ · uᵢ · vᵢᵀ`는 **랭크-1 행렬** 하나입니다.  
k를 작게 할수록 → 압축률 높음, 품질 낮음  
k를 크게 할수록 → 압축률 낮음, 품질 높음

In [ ]:
# ============================================================
# 이미지 압축 실습 (합성 이미지 사용)

In [ ]:
# 실습용 합성 이미지 생성 (체스판 + 노이즈)
np.random.seed(42)
size = 128

# 체스판 패턴 생성
x, y = np.meshgrid(np.arange(size), np.arange(size))
chess = ((x // 16 + y // 16) % 2).astype(float)
# 가우시안 노이즈 추가
noise = np.random.randn(size, size) * 0.1
img = chess + noise  # 최종 이미지 (128x128)

print(f'이미지 행렬 크기: {img.shape}')
print(f'원본 데이터 수: {img.shape[0] * img.shape[1]} 개 숫자')

# SVD 수행
U_img, S_img, Vt_img = np.linalg.svd(img, full_matrices=False)
print(f'\n전체 특이값 개수: {len(S_img)}')
print(f'상위 5개 특이값: {S_img[:5]}')

# 다양한 k값으로 저랭크 근사
k_values = [1, 5, 10, 20, 50, 128]  # 사용할 특이값 개수
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for idx, k in enumerate(k_values):
    # 상위 k개의 특이값만 사용해서 재구성
    # U_img[:, :k] → 처음 k개 열만 선택
    # np.diag(S_img[:k]) → k×k 대각행렬
    # Vt_img[:k, :] → 처음 k개 행만 선택
    A_k = U_img[:, :k] @ np.diag(S_img[:k]) @ Vt_img[:k, :]
    
    # 압축률 계산: 원본 크기 대비 저장해야 할 숫자의 비율
    original_size = size * size  # 128 * 128 = 16384
    compressed_size = k * size + k + k * size  # U 열 + S + Vt 행
    compression_ratio = original_size / compressed_size
    
    # 오차(RMSE) 계산
    rmse = np.sqrt(np.mean((img - A_k)**2))
    
    axes[idx].imshow(A_k, cmap='gray', vmin=0, vmax=1)
    axes[idx].set_title(f'k={k}개 특이값\n압축비={compression_ratio:.1f}x, RMSE={rmse:.3f}', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('저랭크 근사(SVD)를 이용한 이미지 압축\n(k=사용한 특이값 개수)', fontsize=13)
plt.tight_layout()
plt.show()

print('\n▶ k가 작을수록 압축률은 높지만 품질이 떨어집니다.')
print('▶ k=128이면 원본과 동일 (손실 없음)')

---
## 5. 트랜스포머에서 SVD: LoRA (Low-Rank Adaptation) 원리

### LoRA란?

GPT-4나 LLaMA 같은 대형 언어 모델을 파인튜닝(fine-tuning)할 때,  
모든 파라미터를 업데이트하면 비용이 너무 큽니다.

**LoRA의 핵심 아이디어:**  
큰 가중치 행렬 W(예: 4096×4096)를 직접 업데이트하는 대신,  
변화량 ΔW를 **두 개의 작은 행렬 A×B로 근사**합니다:

$$W_{new} = W_{old} + \Delta W \approx W_{old} + B \cdot A$$

여기서:
- A: (rank r) × d 크기 → 작음
- B: d × (rank r) 크기 → 작음
- r ≪ d (예: r=8, d=4096)

**왜 이것이 SVD와 관련?**  
SVD의 저랭크 근사 `A_k = U_k Σ_k Vt_k`에서,  
B = U_k, A = Σ_k Vt_k 로 볼 수 있습니다!

In [ ]:
# ============================================================
# LoRA 원리 시뮬레이션

In [ ]:
np.random.seed(0)

# 트랜스포머의 가중치 행렬 W (실제는 4096x4096, 여기선 64x64로 축소)
d_model = 64   # 모델 차원 (실제로는 수천)
W_pretrained = np.random.randn(d_model, d_model) * 0.02  # 사전학습된 가중치
print(f'사전학습 가중치 행렬 W: {W_pretrained.shape}')
print(f'W의 파라미터 수: {d_model * d_model:,}  (실제 GPT-3: 4096x4096 = 16,777,216개)')

# '파인튜닝으로 학습해야 할' 이상적인 변화량 (보통 이 값을 모름, 학습으로 구함)
# 여기서는 데모를 위해 SVD로 저랭크 구조를 가진 ΔW를 직접 만들겠습니다
rank_true = 4  # 실제 변화량의 랭크 (대부분의 파인튜닝은 저랭크 구조를 가짐)
B_true = np.random.randn(d_model, rank_true)
A_true = np.random.randn(rank_true, d_model)
delta_W_true = B_true @ A_true  # 이것이 '이상적인' 가중치 변화량

W_target = W_pretrained + delta_W_true  # 파인튜닝 후 목표 가중치

print(f'\n이상적인 변화량 ΔW: {delta_W_true.shape}')
print(f'  랭크-{rank_true} 구조: B({d_model}x{rank_true}) × A({rank_true}x{d_model})')

# LoRA 방식: 다양한 rank r로 ΔW를 근사
print('\n=== LoRA: 다양한 rank r로 ΔW 근사 ===')
print(f'{"rank r":<10} {"파라미터 절감":<18} {"근사 오차":<15}')
print('-' * 45)

U_dw, S_dw, Vt_dw = np.linalg.svd(delta_W_true, full_matrices=False)

for r in [1, 2, 4, 8, 16, 64]:
    r_actual = min(r, len(S_dw))
    
    # LoRA 근사: B = U_r * sqrt(Σ_r), A = sqrt(Σ_r) * Vt_r
    sqrt_S = np.sqrt(S_dw[:r_actual])
    B_lora = U_dw[:, :r_actual] * sqrt_S[np.newaxis, :]  # (d x r)
    A_lora = sqrt_S[:, np.newaxis] * Vt_dw[:r_actual, :]  # (r x d)
    delta_W_approx = B_lora @ A_lora
    
    # 파라미터 수 비교
    lora_params = r_actual * d_model + r_actual * d_model  # A + B
    full_params = d_model * d_model
    reduction = (1 - lora_params / full_params) * 100
    
    # 근사 오차
    error = np.linalg.norm(delta_W_true - delta_W_approx, 'fro')
    
    marker = ' ← LoRA 주로 이 범위 사용' if r in [4, 8] else ''
    print(f'r={r:<8} {reduction:.1f}% 절감{"":>5} 오차={error:.4f}{marker}')

print(f'\n▶ 실제 LoRA 논문(Hu et al. 2021)에서는 r=4~16을 권장합니다.')
print('▶ r이 작을수록 파라미터가 적고(빠른 학습), 오차가 큽니다.')

In [ ]:
# ============================================================
# LoRA 구조를 PyTorch 스타일로 구현 (라이브러리 없이)

In [ ]:
class LoRALayer:
    """
    LoRA (Low-Rank Adaptation) 레이어 구현
    
    핵심 아이디어:
    - 원본 가중치 W는 고정 (frozen)
    - ΔW = B @ A 만 학습 (B와 A는 작은 행렬)
    """
    def __init__(self, d_in, d_out, rank=4, alpha=1.0):
        """
        Args:
            d_in: 입력 차원
            d_out: 출력 차원  
            rank: LoRA 랭크 r (작을수록 파라미터 절약)
            alpha: 스케일링 팩터 (보통 rank와 같은 값 사용)
        """
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank  # 출력을 alpha/rank로 스케일
        
        # 원본 가중치 (사전학습된 것, 학습 중 고정)
        self.W = np.random.randn(d_out, d_in) * 0.02
        
        # LoRA 행렬 초기화
        # A: 가우시안 초기화 (논문 방식)
        self.A = np.random.randn(rank, d_in) * 0.02  
        # B: 0으로 초기화 → 학습 시작 시 ΔW = B@A = 0
        self.B = np.zeros((d_out, rank))
        
    def forward(self, x):
        """
        순전파: output = W@x + scaling * B@A@x
        """
        # 원본 W의 출력
        original_output = x @ self.W.T
        
        # LoRA 추가 출력: x → A → B (저차원 공간 경유)
        lora_output = x @ self.A.T @ self.B.T * self.scaling
        
        return original_output + lora_output
    
    def num_params(self):
        W_params = self.W.size
        lora_params = self.A.size + self.B.size
        return W_params, lora_params


# 예제: 64차원 입력/출력, rank=4 LoRA
d = 64
lora = LoRALayer(d_in=d, d_out=d, rank=4, alpha=4.0)

# 더미 입력 (배치 크기=8, 시퀀스 길이 없이 단순 벡터)
x = np.random.randn(8, d)
out = lora.forward(x)

W_params, lora_params = lora.num_params()
print('=== LoRALayer 정보 ===')
print(f'입력 차원: {d}, 출력 차원: {d}, rank: {lora.rank}')
print(f'입력 shape: {x.shape} → 출력 shape: {out.shape}')
print(f'\n원본 W 파라미터: {W_params:,} 개')
print(f'LoRA 파라미터 (A+B): {lora_params:,} 개')
print(f'파라미터 절감: {(1 - lora_params/W_params)*100:.1f}%')
print(f'\n실제 GPT-3 scale (4096x4096)에서 rank=8이면:')
print(f'  W 파라미터: {4096*4096:,}개')
print(f'  LoRA 파라미터: {2*8*4096:,}개')
print(f'  절감률: {(1 - 2*8*4096/(4096*4096))*100:.2f}%')

---
## 6. 트랜스포머 Attention 가중치의 SVD 분석

실제 트랜스포머의 Attention 가중치 행렬을 SVD로 분석하면  
**어떤 정보가 중요하게 학습되었는지** 이해할 수 있습니다.

Attention 공식: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$

여기서 Q, K, V는 각각 가중치 행렬 $W_Q, W_K, W_V$를 통해 만들어집니다.

In [ ]:
# ============================================================
# Attention 가중치 행렬의 SVD 분석

In [ ]:
np.random.seed(42)

# 미니 트랜스포머 설정
seq_len = 10   # 시퀀스 길이 (토큰 수)
d_model = 32   # 임베딩 차원
d_k = 8        # Query/Key 차원
d_v = 8        # Value 차원

# 학습된 가중치 행렬 시뮬레이션 (실제 모델에서는 역전파로 학습됨)
W_Q = np.random.randn(d_model, d_k) * np.sqrt(2.0 / (d_model + d_k))
W_K = np.random.randn(d_model, d_k) * np.sqrt(2.0 / (d_model + d_k))
W_V = np.random.randn(d_model, d_v) * np.sqrt(2.0 / (d_model + d_v))

# 입력 시퀀스 (예: 문장의 단어 임베딩)
X = np.random.randn(seq_len, d_model)  # (10, 32)

# Q, K, V 계산
Q = X @ W_Q  # (10, 8)
K = X @ W_K  # (10, 8)
V = X @ W_V  # (10, 8)

# Attention 점수 행렬 계산 (Scaled Dot-Product)
# Q @ K.T → (10, 10): i번째 토큰이 j번째 토큰에 얼마나 주목하는지
scores = Q @ K.T / np.sqrt(d_k)

# Softmax 적용 → Attention 가중치 (확률 분포)
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

attention_weights = softmax(scores)  # (10, 10)

print('=== Attention 행렬 분석 ===')
print(f'Attention 가중치 행렬 shape: {attention_weights.shape}')
print('(i,j) 위치: i번째 토큰이 j번째 토큰에 주목하는 가중치')
print()
print('Attention 가중치 행렬 (반올림):')
print(attention_weights.round(3))
print(f'각 행의 합: {attention_weights.sum(axis=1).round(3)}  (모두 1.0이어야 함)')

# Attention 행렬의 SVD 분석
U_att, S_att, Vt_att = np.linalg.svd(attention_weights, full_matrices=False)

print(f'\n=== Attention 가중치 행렬의 SVD ===')
print(f'특이값: {S_att.round(4)}')

# 누적 분산 설명량
cumulative = np.cumsum(S_att**2) / np.sum(S_att**2) * 100
print(f'\n누적 분산 설명량:')
for i, (s, c) in enumerate(zip(S_att, cumulative)):
    print(f'  상위 {i+1}개: {c:.1f}%')
    if c > 95:
        print(f'  → 상위 {i+1}개 특이값으로 95% 이상의 정보 설명 가능!')
        break

In [ ]:
# ============================================================
# Attention Map 시각화 + SVD 분석

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) 원본 Attention 맵
im1 = axes[0].imshow(attention_weights, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('원본 Attention 가중치\n(10x10)', fontsize=11)
axes[0].set_xlabel('Key 토큰 위치')
axes[0].set_ylabel('Query 토큰 위치')
plt.colorbar(im1, ax=axes[0])

# 2) 특이값 그래프
axes[1].bar(range(1, len(S_att)+1), S_att, color='steelblue', alpha=0.7)
axes[1].plot(range(1, len(S_att)+1), S_att, 'r-o', markersize=4)
axes[1].set_xlabel('특이값 번호')
axes[1].set_ylabel('특이값 크기')
axes[1].set_title('Attention 행렬의\n특이값 분포', fontsize=11)
axes[1].axhline(y=S_att[0]*0.1, color='green', linestyle='--', label='σ₁의 10%')
axes[1].legend()

# 3) rank-3 근사 Attention 맵
k_approx = 3
att_approx = U_att[:, :k_approx] @ np.diag(S_att[:k_approx]) @ Vt_att[:k_approx, :]
att_approx = np.clip(att_approx, 0, 1)  # 값 범위 클리핑
im3 = axes[2].imshow(att_approx, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title(f'rank-{k_approx} 근사 Attention\n(LoRA 개념과 동일)', fontsize=11)
axes[2].set_xlabel('Key 토큰 위치')
axes[2].set_ylabel('Query 토큰 위치')
plt.colorbar(im3, ax=axes[2])

plt.suptitle('Attention 가중치 행렬의 SVD 분석', fontsize=13)
plt.tight_layout()
plt.show()

print('\n▶ Attention 행렬도 저랭크 구조를 가지는 경우가 많습니다.')
print('▶ 이것이 LoRA가 효과적인 이유 중 하나입니다.')

---
## 7. 연습 문제

아래 문제들을 직접 풀어보세요!

In [ ]:
# ============================================================
# 연습 문제 1: 다음 행렬을 SVD로 분해하고 복원 오차를 계산하세요

In [ ]:
B_exercise = np.array([
    [2, 4, 1, 3],
    [1, 3, 0, 2],
    [0, 1, 2, 1]
], dtype=float)

# TODO: SVD 분해
# U_ex, S_ex, Vt_ex = np.linalg.svd(B_exercise, full_matrices=False)

# TODO: 특이값의 개수와 크기를 출력하세요
# print('특이값:', S_ex)

# TODO: rank-2 근사를 계산하고 원본과의 Frobenius norm 오차를 구하세요
# Frobenius norm: np.linalg.norm(A - A_approx, 'fro')

print('연습 문제 1: 위의 TODO 주석을 풀어서 실행해보세요!')
print('정답 힌트: rank-2 근사의 오차 ≈ sqrt(σ₃²) = 세 번째 특이값의 크기')

In [ ]:
# 연습 문제 2: LoRA rank 선택

In [ ]:
print('\n연습 문제 2:')
print('d=256, 파라미터를 원본의 5% 이하로 줄이면서')
print('사용 가능한 최대 rank r은 얼마인가?')
print('\n풀이: r × d + r × d < 0.05 × d × d')
print('      2rd < 0.05d²')
print('      r < 0.05d/2 = 0.025 × 256 =', 0.025 * 256)
print('      → r ≤ 6')

---
## 📚 핵심 정리

| 개념 | 내용 |
|------|------|
| **SVD** | 행렬 A를 U·Σ·Vᵀ로 분해. 모든 행렬에 적용 가능 |
| **특이값** | 각 방향의 '중요도'. 내림차순으로 정렬됨 |
| **저랭크 근사** | 상위 k개 특이값만 사용해 압축 표현 |
| **LoRA** | SVD의 저랭크 아이디어를 파인튜닝에 적용 |
| **Attention 분석** | Attention 행렬도 저랭크 구조 → LoRA 효과적 |

### 다음 단계
- **059**: 텐서곱과 크로네커 곱
- **LoRA 논문**: Hu et al. "LoRA: Low-Rank Adaptation of Large Language Models" (2021)